# CC3092 – Deep Learning · Proyecto 2
## Detección secuencial de lavado de dinero en transacciones IBM AML

**Curso:** CC3092 – Deep Learning  
**Fecha:** [COMPLETAR]

Este notebook implementa un sistema de detección en dos etapas:

1. **Etapa A:** un LSTM Encoder–Decoder aprende a reconstruir únicamente secuencias normales. El error de reconstrucción se convierte en score de anomalía.
2. **Etapa B:** un clasificador reutiliza el encoder anterior, incorpora atención de producto punto escalado y estima la probabilidad de lavado.

Se incluye una línea base entrenada desde cero, combinación de señales, ablación, interpretabilidad por transacción y exportación de artefactos para el MVP.

> El notebook no contiene métricas inventadas. Debe ejecutarse con los archivos de datos para producir los resultados finales.

## Bloque 0 — Configuración y reproducibilidad

El modo rápido permite verificar el pipeline. Para la ejecución final se debe usar `MODO_RAPIDO = False`. El notebook está preparado para ejecutarse localmente desde la raíz del proyecto y utiliza rutas relativas, de modo que la carpeta completa pueda moverse o compartirse sin modificar rutas absolutas.

In [1]:
from dataclasses import dataclass, asdict
from pathlib import Path
from collections import Counter
import copy, csv, json, math, os, random, time, warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    average_precision_score, roc_auc_score, precision_recall_curve,
    precision_score, recall_score, f1_score, confusion_matrix
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

@dataclass
class Config:
    SEED: int = 42
    MODO_RAPIDO: bool = False
    DATA_DIR: str = '.'
    IBM_FILE: str = 'ibm-transactions/HI-Small_Trans.csv'
    PATTERNS_FILE: str = 'ibm-transactions/HI-Small_Patterns.txt'
    PAYSIM_FILE: str = 'paysim/PS_20174392719_1491204439457_log.csv'
    ARTIFACT_DIR: str = 'artifacts_aml'
    MIN_SEQ_LEN: int = 3
    MAX_SEQ_LEN: int = 30
    MAX_WINDOWS_PER_SENDER: int = 20
    MAX_NORMAL_AE: int = 60000
    NORMAL_TO_POSITIVE_B: int = 10
    BATCH_SIZE: int = 256
    HIDDEN_SIZE: int = 64
    LATENT_SIZE: int = 64
    ATTN_SIZE: int = 32
    DROPOUT: float = 0.20
    AE_EPOCHS: int = 8
    HEAD_EPOCHS: int = 3
    FINETUNE_EPOCHS: int = 6
    LR_AE: float = 1e-3
    LR_HEAD: float = 1e-3
    LR_ENCODER: float = 1e-4
    PATIENCE: int = 3
    NUM_WORKERS: int = 0

CFG = Config()
if CFG.MODO_RAPIDO:
    CFG.MAX_NORMAL_AE = 8000
    CFG.AE_EPOCHS = 2
    CFG.HEAD_EPOCHS = 1
    CFG.FINETUNE_EPOCHS = 2

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_seed(CFG.SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
PROJECT_ROOT = Path.cwd().resolve()
DATA_DIR = (PROJECT_ROOT / CFG.DATA_DIR).resolve()
IBM_PATH = DATA_DIR / CFG.IBM_FILE
PATTERNS_PATH = DATA_DIR / CFG.PATTERNS_FILE
PAYSIM_PATH = DATA_DIR / CFG.PAYSIM_FILE
ARTIFACT_DIR = (PROJECT_ROOT / CFG.ARTIFACT_DIR).resolve()
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
T0_GLOBAL = time.time()

print('PyTorch:', torch.__version__)
print('Pandas:', pd.__version__)
print('Dispositivo:', DEVICE)
print('Raíz del proyecto:', PROJECT_ROOT)
print('Archivo IBM:', IBM_PATH)
print('Archivo de patrones:', PATTERNS_PATH)
print('Directorio de artefactos:', ARTIFACT_DIR)
print('Modo rápido:', CFG.MODO_RAPIDO)

PyTorch: 2.14.0+cpu
Pandas: 2.3.3
Dispositivo: cpu
Raíz del proyecto: C:\Users\maria\OneDrive\Documents\UVG\Deep Learning\Proyecto 2
Archivo IBM: C:\Users\maria\OneDrive\Documents\UVG\Deep Learning\Proyecto 2\ibm-transactions\HI-Small_Trans.csv
Archivo de patrones: C:\Users\maria\OneDrive\Documents\UVG\Deep Learning\Proyecto 2\ibm-transactions\HI-Small_Patterns.txt
Directorio de artefactos: C:\Users\maria\OneDrive\Documents\UVG\Deep Learning\Proyecto 2\artifacts_aml
Modo rápido: False
